# 01 — Data Generation
## Ma3 | Synthetic Training Data for Nairobi Matatu Network

This notebook generates three datasets that will train our three ML models:

| Dataset | File | Used by |
|---|---|---|
| GPS traces + ETA labels | `data/eta_dataset.csv` | `02_eta_model.ipynb` |
| Hourly ridership per route | `data/demand_dataset.csv` | `03_demand_forecast.ipynb` |
| Driver telemetry + anomaly labels | `data/driver_dataset.csv` | `04_driver_scoring.ipynb` |

All data is synthetic but designed around real Nairobi patterns:
- Peak hours: 6–9 AM, 5–8 PM
- Routes: 46 CBD-Westlands, 34 CBD-Kangemi, 58 CBD-Kikuyu
- Matatu capacity: 14 passengers

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random
import os

random.seed(42)
np.random.seed(42)

# Output directory
DATA_DIR = "../data"
os.makedirs(DATA_DIR, exist_ok=True)

# Routes
ROUTES = ["46 CBD-Westlands", "34 CBD-Kangemi", "58 CBD-Kikuyu"]

# Stops per route
STOPS = {
    "46 CBD-Westlands": ["CBD", "Kencom", "University Way", "Museum Hill", "Westlands"],
    "34 CBD-Kangemi":   ["CBD", "GPO", "Kawangware", "Dagoretti", "Kangemi"],
    "58 CBD-Kikuyu":    ["CBD", "Muthurwa", "Kawangware", "Ruthimitu", "Kikuyu"],
}

# Base fares
FARES = {"46 CBD-Westlands": 50, "34 CBD-Kangemi": 60, "58 CBD-Kikuyu": 80}

print("Constants loaded ✓")
print(f"Routes: {ROUTES}")
print(f"Stops: {STOPS}")
print(f"Fares: {FARES}")

Constants loaded ✓
Routes: ['46 CBD-Westlands', '34 CBD-Kangemi', '58 CBD-Kikuyu']
Stops: {'46 CBD-Westlands': ['CBD', 'Kencom', 'University Way', 'Museum Hill', 'Westlands'], '34 CBD-Kangemi': ['CBD', 'GPO', 'Kawangware', 'Dagoretti', 'Kangemi'], '58 CBD-Kikuyu': ['CBD', 'Muthurwa', 'Kawangware', 'Ruthimitu', 'Kikuyu']}
Fares: {'46 CBD-Westlands': 50, '34 CBD-Kangemi': 60, '58 CBD-Kikuyu': 80}


## Part 1 — ETA Dataset

Each row represents one matatu at a given stop, at a given time.

**Features:**
- `hour` — hour of day (0–23)
- `day_of_week` — 0=Monday, 6=Sunday
- `stop_sequence` — position along route (1–5)
- `pax_count` — passengers on board
- `speed_kmh` — current speed
- `is_peak` — 1 if rush hour

**Target:**
- `eta_minutes` — minutes to next stop

In [2]:
def is_peak(hour):
    return 1 if (6 <= hour <= 9) or (17 <= hour <= 20) else 0

def eta_minutes(hour, stop_seq, pax_count, speed):
    """Simulate realistic ETA based on traffic and load."""
    base = 3 + stop_seq * 1.5
    traffic = 1.8 if is_peak(hour) else 1.0
    load = 1 + (pax_count / 14) * 0.3
    noise = np.random.normal(0, 0.8)
    return max(1, round(base * traffic * load + noise, 1))

rows = []
# Simulate 60 days × 18 hours × 3 routes × 5 stops × 3 vehicles
for day in range(60):
    dow = day % 7
    for hour in range(6, 24):
        for route in ROUTES:
            stops = STOPS[route]
            for vehicle in range(3):
                pax = random.randint(2, 14)
                speed = random.uniform(10, 20) if is_peak(hour) else random.uniform(25, 55)
                for seq, stop in enumerate(stops[:-1], start=1):
                    rows.append({
                        "route": route,
                        "stop_name": stop,
                        "stop_sequence": seq,
                        "hour": hour,
                        "day_of_week": dow,
                        "pax_count": pax,
                        "speed_kmh": round(speed, 1),
                        "is_peak": is_peak(hour),
                        "eta_minutes": eta_minutes(hour, seq, pax, speed)
                    })
                    pax = max(0, pax + random.randint(-3, 3))

eta_df = pd.DataFrame(rows)
eta_df.to_csv(f"{DATA_DIR}/eta_dataset.csv", index=False)

print(f"ETA dataset: {eta_df.shape[0]:,} rows × {eta_df.shape[1]} cols")
print(eta_df.head())

ETA dataset: 38,880 rows × 9 cols
              route       stop_name  stop_sequence  hour  day_of_week  \
0  46 CBD-Westlands             CBD              1     6            0   
1  46 CBD-Westlands          Kencom              2     6            0   
2  46 CBD-Westlands  University Way              3     6            0   
3  46 CBD-Westlands     Museum Hill              4     6            0   
4  46 CBD-Westlands             CBD              1     6            0   

   pax_count  speed_kmh  is_peak  eta_minutes  
0         12       11.1        1         10.6  
1         14       11.1        1         13.9  
2         13       11.1        1         17.8  
3         11       11.1        1         21.2  
4          4       17.4        1          8.6  


## Part 2 — Demand Dataset

Each row represents ridership on a route for a given hour slot over 90 days.

**Features:**
- `hour`, `day_of_week`, `is_peak`
- `is_weekend`, `is_holiday`
- `route`

**Target:**
- `pax_demand` — total passengers that hour on that route

In [3]:
HOLIDAYS = [0, 25, 60]  # day indices treated as public holidays

def demand(hour, dow, is_holiday):
    base = 80 if is_peak(hour) else 30
    weekend_factor = 0.6 if dow >= 5 else 1.0
    holiday_factor = 0.4 if is_holiday else 1.0
    noise = np.random.normal(0, 5)
    return max(5, int(base * weekend_factor * holiday_factor + noise))

rows = []
for day in range(90):
    dow = day % 7
    is_holiday = 1 if day in HOLIDAYS else 0
    for hour in range(5, 23):
        for route in ROUTES:
            rows.append({
                "route": route,
                "hour": hour,
                "day_of_week": dow,
                "is_peak": is_peak(hour),
                "is_weekend": 1 if dow >= 5 else 0,
                "is_holiday": is_holiday,
                "pax_demand": demand(hour, dow, is_holiday)
            })

demand_df = pd.DataFrame(rows)
demand_df.to_csv(f"{DATA_DIR}/demand_dataset.csv", index=False)

print(f"Demand dataset: {demand_df.shape[0]:,} rows × {demand_df.shape[1]} cols")
print(demand_df.head())

Demand dataset: 4,860 rows × 7 cols
              route  hour  day_of_week  is_peak  is_weekend  is_holiday  \
0  46 CBD-Westlands     5            0        0           0           1   
1    34 CBD-Kangemi     5            0        0           0           1   
2     58 CBD-Kikuyu     5            0        0           0           1   
3  46 CBD-Westlands     6            0        1           0           1   
4    34 CBD-Kangemi     6            0        1           0           1   

   pax_demand  
0          12  
1          13  
2          15  
3          27  
4          37  


## Part 3 — Driver Telemetry Dataset

Each row is one trip by one driver. Anomalous drivers are injected at ~10% rate — 
these represent dangerous or irregular behaviour (speeding, off-route, long idles).

**Features:**
- `speed_variance` — how erratic the speed was
- `off_route_ratio` — fraction of trip spent off designated route
- `avg_dwell_time` — average seconds idling at stops
- `harsh_braking_events` — count of sudden stops
- `trip_duration_min` — total trip time

**Label:**
- `is_anomaly` — 1 = flagged driver behaviour, 0 = normal

In [4]:
rows = []
N_DRIVERS = 20
N_TRIPS = 150  # trips per driver

for driver_id in range(N_DRIVERS):
    # 10% of drivers are "bad"
    is_bad_driver = driver_id < int(N_DRIVERS * 0.10)

    for trip in range(N_TRIPS):
        if is_bad_driver:
            speed_var      = np.random.uniform(30, 80)
            off_route      = np.random.uniform(0.2, 0.6)
            dwell          = np.random.uniform(60, 180)
            harsh_braking  = np.random.randint(5, 15)
            duration       = np.random.uniform(40, 90)
            anomaly        = 1
        else:
            speed_var      = np.random.uniform(2, 20)
            off_route      = np.random.uniform(0.0, 0.08)
            dwell          = np.random.uniform(10, 45)
            harsh_braking  = np.random.randint(0, 3)
            duration       = np.random.uniform(15, 40)
            anomaly        = 0

        rows.append({
            "driver_id": f"DRV_{driver_id:03d}",
            "speed_variance": round(speed_var, 2),
            "off_route_ratio": round(off_route, 3),
            "avg_dwell_time": round(dwell, 1),
            "harsh_braking_events": harsh_braking,
            "trip_duration_min": round(duration, 1),
            "is_anomaly": anomaly
        })

driver_df = pd.DataFrame(rows)
driver_df.to_csv(f"{DATA_DIR}/driver_dataset.csv", index=False)

print(f"Driver dataset: {driver_df.shape[0]:,} rows × {driver_df.shape[1]} cols")
print(f"Anomaly rate: {driver_df.is_anomaly.mean():.1%}")
print(driver_df.head())

Driver dataset: 3,000 rows × 7 cols
Anomaly rate: 10.0%
  driver_id  speed_variance  off_route_ratio  avg_dwell_time  \
0   DRV_000           77.90            0.372           110.0   
1   DRV_000           53.86            0.514           154.0   
2   DRV_000           34.73            0.590            75.5   
3   DRV_000           53.67            0.363            81.7   
4   DRV_000           73.28            0.449           133.8   

   harsh_braking_events  trip_duration_min  is_anomaly  
0                     7               73.9           1  
1                    14               56.1           1  
2                     7               80.9           1  
3                     8               56.6           1  
4                    10               46.2           1  


In [5]:
print("=" * 45)
print("DATASETS SAVED TO ml/data/")
print("=" * 45)
for fname in ["eta_dataset.csv", "demand_dataset.csv", "driver_dataset.csv"]:
    path = f"{DATA_DIR}/{fname}"
    df = pd.read_csv(path)
    print(f"  {fname:<25} {df.shape[0]:>6,} rows  {df.shape[1]:>2} cols")
print("=" * 45)
print("Ready for 02_eta_model.ipynb ✓")

DATASETS SAVED TO ml/data/
  eta_dataset.csv           38,880 rows   9 cols
  demand_dataset.csv         4,860 rows   7 cols
  driver_dataset.csv         3,000 rows   7 cols
Ready for 02_eta_model.ipynb ✓
